<a href="https://colab.research.google.com/github/lovnishverma/Python-Getting-Started/blob/main/web_crawling_scrapy_colab_ready.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Web Crawling and Scrapy - Google Colab Ready

*This notebook teaches:*

1. Basic Web Crawling using `requests` + `BeautifulSoup`
2. Professional Crawling using `Scrapy`
3. Saving scraped data to CSV
4. Running Scrapy properly inside Google Colab


In [1]:
# Install required libraries
!pip install scrapy beautifulsoup4 requests pandas -q

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 46.3/46.3 kB 2.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 352.9/352.9 kB 12.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 57.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 269.8/269.8 kB 18.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 105.9/105.9 kB 7.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.8/42.8 kB 2.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 74.6/74.6 kB 4.8 MB/s eta 0:00:00


# **Part 1: Basic Web Crawler**

This crawler:
- Visits a webpage
- Extracts links
- Crawls discovered pages


In [2]:
import requests
from bs4 import BeautifulSoup
from urllib.parse import urljoin

# Starting URL
url = "https://example.com"

visited = set()
to_visit = [url]

max_pages = 5

while to_visit and len(visited) < max_pages:

    current_url = to_visit.pop(0)

    if current_url in visited:
        continue

    try:
        print(f"\\nCrawling: {current_url}")

        response = requests.get(current_url, timeout=5)

        soup = BeautifulSoup(response.text, "html.parser")

        visited.add(current_url)

        links = soup.find_all("a", href=True)

        for link in links:
            full_url = urljoin(current_url, link["href"])

            if full_url not in visited:
                to_visit.append(full_url)

    except Exception as e:
        print("Error:", e)

print("\\nVisited Pages:")

for page in visited:
    print(page)

\nCrawling: https://example.com
\nCrawling: https://iana.org/domains/example
\nCrawling: https://iana.org/
\nCrawling: https://iana.org/domains
\nCrawling: https://iana.org/protocols
\nVisited Pages:
https://iana.org/domains
https://iana.org/protocols
https://iana.org/domains/example
https://example.com
https://iana.org/


# **Part 2: Scrapy Web Crawler**

This is the correct Colab-ready approach.

We use:
- `CrawlerProcess`
- No Scrapy project required
- No `scrapy crawl` terminal command




---



In [2]:
!pip install scrapy nest_asyncio -q

In [1]:
import nest_asyncio
nest_asyncio.apply()

import scrapy
from scrapy.crawler import CrawlerProcess
import pandas as pd


class QuotesSpider(scrapy.Spider):

    name = "quotes"

    start_urls = [
        "https://quotes.toscrape.com"
    ]

    custom_settings = {
        "FEEDS": {
            "quotes.csv": {
                "format": "csv",
                "overwrite": True
            }
        },
        "LOG_LEVEL": "ERROR"
    }

    def parse(self, response):

        quotes = response.css(".quote")

        for quote in quotes:

            yield {
                "text": quote.css(".text::text").get(),
                "author": quote.css(".author::text").get(),
                "tags": ",".join(
                    quote.css(".tag::text").getall()
                )
            }

        next_page = response.css(
            "li.next a::attr(href)"
        ).get()

        if next_page:
            yield response.follow(
                next_page,
                callback=self.parse
            )


# Create crawler process
process = CrawlerProcess()

# Add spider
process.crawl(QuotesSpider)

# Start crawling
process.start()

print("Crawling Finished!")

INFO:scrapy.utils.log:Scrapy 2.15.2 started (bot: scrapybot)
2026-05-18 04:23:07 [scrapy.utils.log] INFO: Scrapy 2.15.2 started (bot: scrapybot)
INFO:scrapy.utils.log:Versions:
{'lxml': '6.1.0',
 'libxml2': '2.14.6',
 'cssselect': '1.4.0',
 'parsel': '1.11.0',
 'w3lib': '2.4.1',
 'Twisted': '25.5.0',
 'Python': '3.12.13 (main, Mar  4 2026, 09:23:07) [GCC 11.4.0]',
 'pyOpenSSL': '24.2.1 (OpenSSL 3.3.2 3 Sep 2024)',
 'cryptography': '43.0.3',
 'Platform': 'Linux-6.6.122+-x86_64-with-glibc2.35'}
2026-05-18 04:23:07 [scrapy.utils.log] INFO: Versions:
{'lxml': '6.1.0',
 'libxml2': '2.14.6',
 'cssselect': '1.4.0',
 'parsel': '1.11.0',
 'w3lib': '2.4.1',
 'Twisted': '25.5.0',
 'Python': '3.12.13 (main, Mar  4 2026, 09:23:07) [GCC 11.4.0]',
 'pyOpenSSL': '24.2.1 (OpenSSL 3.3.2 3 Sep 2024)',
 'cryptography': '43.0.3',
 'Platform': 'Linux-6.6.122+-x86_64-with-glibc2.35'}
DEBUG:scrapy.crawler:Using CrawlerProcess
2026-05-18 04:23:07 [scrapy.crawler] DEBUG: Using CrawlerProcess
INFO:scrapy.addons:

Crawling Finished!


# View Scraped Data

In [3]:
import pandas as pd

df = pd.read_csv("quotes.csv")

print(df.head())

                                                text           author  \
0  “The world as we have created it is a process ...  Albert Einstein   
1  “It is our choices, Harry, that show what we t...     J.K. Rowling   
2  “There are only two ways to live your life. On...  Albert Einstein   
3  “The person, be it gentleman or lady, who has ...      Jane Austen   
4  “Imperfection is beauty, madness is genius and...   Marilyn Monroe   

                                       tags  
0       change,deep-thoughts,thinking,world  
1                         abilities,choices  
2  inspirational,life,live,miracle,miracles  
3             aliteracy,books,classic,humor  
4                 be-yourself,inspirational  


# Download CSV File

In [4]:
from google.colab import files

files.download("quotes.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>